# O que precisamos para criar um `grafo` no `LangGraph`
- **State**: Define o estado do grafo (pode ser um `TypedDict`, uma `dataclass` ou um modelo `Pydantic`)
- **Nodes**: Funções que recebem o estado como input, executam ações e retornam o `estado` atualizado.
- **Edges**: Conexões entre nós, podendo ser simples ou condicionais.

In [39]:
from operator import add
from typing import Annotated, Literal
from dataclasses import dataclass
from langgraph.graph import END, START, StateGraph

## Grafo 1

In [38]:
# estado do meu grafo
@dataclass
class State:
    nodes_path: Annotated[list[str], add]
    current_number: int = 0

In [54]:
# definindo os nodes
def node_a(state: State) -> State:
    output_state = State(
        nodes_path=['A'],
        current_number=state.current_number
    )
    print(f'> node_a: {state=} | {output_state=}')
    return output_state
    
def node_b(state: State) -> State:
    output_state = State(
        nodes_path=['B'],
        current_number=state.current_number
    )
    print(f'> node_b: {state=} | {output_state=}')
    return output_state

def node_c(state: State) -> State:
    output_state = State(
        nodes_path=['C'],
        current_number=state.current_number
    )
    print(f'> node_c: {state=} | {output_state=}')
    return output_state

In [55]:
def the_conditional(state: State) -> Literal['goes_to_b', 'goes_to_c']:
    if state.current_number >= 50:
        return 'goes_to_b'
    return 'goes_to_c'

In [56]:
# define o builder do grafo
builder = StateGraph(State)

builder.add_node('A', node_a)
builder.add_node('B', node_b)
builder.add_node('C', node_c)

# conectando as edges (arestas)
builder.add_edge(START, 'A')
# builder.add_conditional_edges('A', the_conditional, ['B', 'C'])
builder.add_conditional_edges(
    'A', the_conditional, {'goes_to_b': 'B', 'goes_to_c': 'C'}
)
builder.add_edge('B', END)
builder.add_edge('C', END)

# builder.add_edge('__start__', 'A')  # Você pode usar START
# builder.add_edge('A', 'B')
# builder.add_edge('B', END)  # você pode usar '__end__'

# compila o grafo
graph = builder.compile()

In [57]:
print(graph.get_graph().draw_ascii())

 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
     +---+       
     | A |       
     +---+       
     .    .      
    .      .     
   .        .    
+---+     +---+  
| B |     | C |  
+---+     +---+  
     *    *      
      *  *       
       **        
  +---------+    
  | __end__ |    
  +---------+    


In [59]:
# seu graph tem todos os métodos que seu llm teria
response = graph.invoke(State(nodes_path=[]))
print(response)

> node_a: state=State(nodes_path=[], current_number=0) | output_state=State(nodes_path=['A'], current_number=0)
> node_c: state=State(nodes_path=['A'], current_number=0) | output_state=State(nodes_path=['C'], current_number=0)
{'nodes_path': ['A', 'C'], 'current_number': 0}


In [60]:
response = graph.invoke(State(nodes_path=[], current_number=50))
print(response)

> node_a: state=State(nodes_path=[], current_number=50) | output_state=State(nodes_path=['A'], current_number=50)
> node_b: state=State(nodes_path=['A'], current_number=50) | output_state=State(nodes_path=['B'], current_number=50)
{'nodes_path': ['A', 'B'], 'current_number': 50}
